In [4]:
filtered = [
    {"stage": "verify", "gene": "OsMYB110", "prompt_tokens": 40689},
    {"stage": "verify", "gene": "OsMYB55",  "prompt_tokens": 40697},
    {"stage": "verify", "gene": "OsDREB2A", "prompt_tokens": 41137},
]
t=sum(c['prompt_tokens'] for c in filtered)
print(t)

122523


In [15]:
calls = [
    {"stage": "extract", "total_tokens": 2300},
    {"stage": "extract", "total_tokens": 3200},
    {"stage": "verify",  "total_tokens": 3500},
    {'stage':'pig'}
]

# 列表推导式写法
extract_calls=[c for c in calls if c['stage']=='extract']
print(extract_calls)

[{'stage': 'extract', 'total_tokens': 2300}, {'stage': 'extract', 'total_tokens': 3200}]


In [16]:
stages=sorted(set(c['stage'] for c in calls))
print(stages)

['extract', 'pig', 'verify']


In [10]:
import json
with open('/data/haotianwu/biojson/reports/verify_tokens/token_usage_verify_20260314_170657.json', 'r') as f:
    data=json.load(f)
print(json.dumps(data, indent=4, ensure_ascii=False))

{
    "timestamp": "2026-03-14T17:06:57.357611",
    "model": "Vendor2/Claude-4.6-opus",
    "calls": [
        {
            "stage": "verify",
            "file": "OsMYB110",
            "prompt_tokens": 40689,
            "completion_tokens": 3288,
            "total_tokens": 43977,
            "timestamp": "2026-03-14T17:05:18.118930",
            "gene": "OsMYB110"
        },
        {
            "stage": "verify",
            "file": "OsMYB55",
            "prompt_tokens": 40697,
            "completion_tokens": 3136,
            "total_tokens": 43833,
            "timestamp": "2026-03-14T17:06:07.942145",
            "gene": "OsMYB55"
        },
        {
            "stage": "verify",
            "file": "OsDREB2A",
            "prompt_tokens": 41137,
            "completion_tokens": 3092,
            "total_tokens": 44229,
            "timestamp": "2026-03-14T17:06:57.339472",
            "gene": "OsDREB2A"
        }
    ],
    "summary": {
        "verify": {
            "pr

In [34]:
from datetime import datetime
now=datetime.now()
a=now
print(now.replace(microsecond=0))

2026-03-15 15:53:00


In [38]:
print(f"{(datetime.now()-a).total_seconds():.2f}")

283.96


In [ ]:
class a:
    def __init__(self):
        self.calls = [
        {"stage": "extract", 'model': 'gpt-3.5-turbo', "prompt_tokens": 2300, "completion_tokens": 500, "total_tokens": 2800,'total_tokens': 2800},
        {"stage": "extract", "prompt_tokens": 3200, "completion_tokens": 600, "total_tokens": 3800,'total_tokens': 3800},
        {"stage": "verify",  "prompt_tokens": 3500, "completion_tokens": 700, "total_tokens": 4200,'total_tokens': 4200}
    ]
    def _aggregate(self, stage_filter=None):
        """按阶段聚合统计"""
        filtered = self.calls if stage_filter is None else [
            c for c in self.calls if c["stage"] == stage_filter
        ]
        prompt = sum(c["prompt_tokens"] for c in filtered)
        completion = sum(c["completion_tokens"] for c in filtered)
        total = sum(c["total_tokens"] for c in filtered)
        return {
            "prompt_tokens": prompt,
            "completion_tokens": completion,
            "total_tokens": total,
            "prompt_ktokens": round(prompt / 1000, 2),
            "completion_ktokens": round(completion / 1000, 2),
            "total_ktokens": round(total / 1000, 2),
            "calls": len(filtered),
        }
    def get_summary(self):
        """获取完整的汇总数据"""
        stages = sorted(set(c["stage"] for c in self.calls))
        summary = {}
        for stage in stages:
            summary[stage] = self._aggregate(stage)
        summary["total"] = self._aggregate()
        return summary

In [28]:
b=a()
summary = b.get_summary()
stages = [s for s in summary if s != "total"]
print(stages)

['extract', 'verify']


In [4]:
"""
token_tracker.py
追踪 API Token 用量，支持多次调用累计，以 kTokens 为单位汇总。

用法:
    from token_tracker import TokenTracker
    tracker = TokenTracker(model="Vendor2/Claude-4.6-opus")
    
    # API 调用后
    response = client.chat.completions.create(...)
    tracker.add(response, stage="extract", file="xxx.md")
    
    # 脚本结束时
    tracker.print_summary()
    tracker.save_report("reports/token_usage_xxx.json")
"""

import json
import os
from datetime import datetime


class TokenTracker:
    """追踪 API Token 用量"""

    def __init__(self, model="unknown"):
        self.model = model
        self.calls = []  # 每次调用的详细记录

    def add(self, response, stage="unknown", file="", gene=""):
        """
        从 API response 中提取 token 用量并记录。
        
        Args:
            response: OpenAI API 返回的 response 对象
            stage: 阶段标识，如 "extract" 或 "verify"
            file: 处理的文件名
            gene: 验证的基因名（仅 verify 阶段使用）
        """
        usage = getattr(response, "usage", None)
        if usage is None:
            print(f"  ⚠️  Token 用量不可用 (stage={stage}, file={file})")
            return

        record = {
            "stage": stage,
            "file": file,
            "prompt_tokens": usage.prompt_tokens or 0,
            "completion_tokens": usage.completion_tokens or 0,
            "total_tokens": usage.total_tokens or 0,
            "timestamp": datetime.now().isoformat(),
        }
        if gene:
            record["gene"] = gene

        self.calls.append(record)

    def _aggregate(self, stage_filter=None):
        """按阶段聚合统计"""
        filtered = self.calls if stage_filter is None else [
            c for c in self.calls if c["stage"] == stage_filter
        ]
        prompt = sum(c["prompt_tokens"] for c in filtered)
        completion = sum(c["completion_tokens"] for c in filtered)
        total = sum(c["total_tokens"] for c in filtered)
        return {
            "prompt_tokens": prompt,
            "completion_tokens": completion,
            "total_tokens": total,
            "prompt_ktokens": round(prompt / 1000, 2),
            "completion_ktokens": round(completion / 1000, 2),
            "total_ktokens": round(total / 1000, 2),
            "calls": len(filtered),
        }

    def get_summary(self):
        """获取完整的汇总数据"""
        stages = sorted(set(c["stage"] for c in self.calls))
        summary = {}
        for stage in stages:
            summary[stage] = self._aggregate(stage)
        summary["total"] = self._aggregate()
        return summary

    def print_summary(self):
        """打印格式化的用量汇总"""
        if not self.calls:
            print("\n📊 Token 用量: 无 API 调用记录")
            return

        summary = self.get_summary()
        stages = [s for s in summary if s != "total"]

        print(f"\n{'═' * 60}")
        print(f"📊 API Token 用量汇总 (模型: {self.model})")
        print(f"{'═' * 60}")
        print(f"  {'阶段':<12} {'输入 (kT)':>10} {'输出 (kT)':>10} {'合计 (kT)':>10} {'调用次数':>8}")
        print(f"  {'─' * 12} {'─' * 10} {'─' * 10} {'─' * 10} {'─' * 8}")

        for stage in stages:
            s = summary[stage]
            print(f"  {stage:<12} {s['prompt_ktokens']:>10.2f} {s['completion_ktokens']:>10.2f} {s['total_ktokens']:>10.2f} {s['calls']:>8}")

        if len(stages) > 1:
            print(f"  {'─' * 12} {'─' * 10} {'─' * 10} {'─' * 10} {'─' * 8}")

        t = summary["total"]
        print(f"  {'总计':<12} {t['prompt_ktokens']:>10.2f} {t['completion_ktokens']:>10.2f} {t['total_ktokens']:>10.2f} {t['calls']:>8}")
        print(f"{'═' * 60}")

    def save_report(self, path):
        """保存详细用量报告为 JSON"""
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)

        report = {
            "timestamp": datetime.now().isoformat(),
            "model": self.model,
            "calls": self.calls,
            "summary": self.get_summary(),
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)

        print(f"  💾 Token 用量报告已保存: {path}")
        return path

    def merge(self, other):
        """合并另一个 TokenTracker 的记录"""
        if isinstance(other, TokenTracker):
            self.calls.extend(other.calls)


# ─── 全局单例（方便跨模块共享）─────────────────────────────────────────────────
_global_tracker = None


def get_tracker(model="unknown"):
    """获取全局 TokenTracker 单例"""
    global _global_tracker
    if _global_tracker is None:
        _global_tracker = TokenTracker(model=model)
    return _global_tracker


def reset_tracker():
    """重置全局 tracker"""
    global _global_tracker
    _global_tracker = None

In [6]:
print(f"\n{'═' * 60}")
print(f"📊 API Token 用量汇总 (模型: {tracker.model})")
print(f"{'═' * 60}")
print(f"  {'阶段':<12} {'输入 (kT)':>10} {'输出 (kT)':>10} {'合计 (kT)':>10} {'调用次数':>8}")
print(f"  {'─' * 12} {'─' * 10} {'─' * 10} {'─' * 10} {'─' * 8}")



════════════════════════════════════════════════════════════
📊 API Token 用量汇总 (模型: gpt-4o)
════════════════════════════════════════════════════════════
  阶段              输入 (kT)    输出 (kT)    合计 (kT)     调用次数
  ──────────── ────────── ────────── ────────── ────────


In [19]:
print(f"\n{'═' * 60}")
print(f"📊 API Token 用量汇总 (模型: {tracker.model})")
print(f"{'═' * 60}")
print(f"  {'phrase':<12} {'input (kT)':>10} {'output (kT)':>10} {'合计 (kT)':>10} {'调用次数':>8}")
print(f"  {'─' * 12} {'─' * 10} {'─' * 10} {'─' * 10} {'─' * 8}")

summary = tracker.get_summary()
stages = [s for s in summary if s != "total"]
for stage in stages:
    s = summary[stage]
    print(f"{stage:>12}{s['prompt_ktokens']:<10.3f}")
    print(f"  {stage:<12} {s['prompt_ktokens']:>10.2f} {s['completion_ktokens']:>10.2f} {s['total_ktokens']:>10.2f} {s['calls']:>8}")
if len(stages) > 1:
    print(f"  {'─' * 12} {'─' * 10} {'─' * 10} {'─' * 10} {'─' * 8}")


════════════════════════════════════════════════════════════
📊 API Token 用量汇总 (模型: gpt-4o)
════════════════════════════════════════════════════════════
  phrase       input (kT) output (kT)    合计 (kT)     调用次数
  ──────────── ────────── ────────── ────────── ────────
     extract1.200     
  extract            1.20       0.55       1.75        1
      verify0.800     
  verify             0.80       0.40       1.20        1
  ──────────── ────────── ────────── ────────── ────────


In [22]:
print(json.dumps(tracker.calls, indent=4, ensure_ascii=False))

[
    {
        "stage": "extract",
        "file": "study_paper_01.md",
        "prompt_tokens": 1200,
        "completion_tokens": 550,
        "total_tokens": 1750,
        "timestamp": "2026-03-15T15:13:23.144240"
    },
    {
        "stage": "verify",
        "file": "study_paper_01.md",
        "prompt_tokens": 800,
        "completion_tokens": 400,
        "total_tokens": 1200,
        "timestamp": "2026-03-15T15:13:23.144294",
        "gene": "BRCA1"
    }
]


In [25]:
report = {
    "timestamp": datetime.now().isoformat(),
    "model": tracker.model,
    "calls": tracker.calls,
    "summary": tracker.get_summary(),
}
print(json.dumps(report, indent=4, ensure_ascii=False))

{
    "timestamp": "2026-03-15T15:47:49.570620",
    "model": "gpt-4o",
    "calls": [
        {
            "stage": "extract",
            "file": "study_paper_01.md",
            "prompt_tokens": 1200,
            "completion_tokens": 550,
            "total_tokens": 1750,
            "timestamp": "2026-03-15T15:13:23.144240"
        },
        {
            "stage": "verify",
            "file": "study_paper_01.md",
            "prompt_tokens": 800,
            "completion_tokens": 400,
            "total_tokens": 1200,
            "timestamp": "2026-03-15T15:13:23.144294",
            "gene": "BRCA1"
        }
    ],
    "summary": {
        "extract": {
            "prompt_tokens": 1200,
            "completion_tokens": 550,
            "total_tokens": 1750,
            "prompt_ktokens": 1.2,
            "completion_ktokens": 0.55,
            "total_ktokens": 1.75,
            "calls": 1
        },
        "verify": {
            "prompt_tokens": 800,
            "completion_tok

In [2]:
from dataclasses import dataclass

# 1. 模拟 OpenAI 的 Usage 结构
@dataclass
class MockUsage:
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

# 2. 模拟整个 Response 对象
@dataclass
class MockResponse:
    usage: MockUsage

# 3. 创建一个真实的测试数据
# 假设这次调用花了 1200 prompt, 500 completion
mock_api_response = MockResponse(
    usage=MockUsage(
        prompt_tokens=1200,
        completion_tokens=550,
        total_tokens=1750
    )
)

In [5]:
# 假设你已经把 TokenTracker 类放在了上面
tracker = TokenTracker(model="gpt-4o")

# 模拟第一步：提取基因
tracker.add(mock_api_response, stage="extract", file="study_paper_01.md")

# 模拟第二步：验证（再来一个不同的数据）
another_response = MockResponse(
    usage=MockUsage(prompt_tokens=800, completion_tokens=400, total_tokens=1200)
)
tracker.add(another_response, stage="verify", file="study_paper_01.md", gene="BRCA1")

# 打印漂亮的汇总表
tracker.print_summary()

# 保存报告
tracker.save_report("test_report.json")


════════════════════════════════════════════════════════════
📊 API Token 用量汇总 (模型: gpt-4o)
════════════════════════════════════════════════════════════
  阶段              输入 (kT)    输出 (kT)    合计 (kT)     调用次数
  ──────────── ────────── ────────── ────────── ────────
  extract            1.20       0.55       1.75        1
  verify             0.80       0.40       1.20        1
  ──────────── ────────── ────────── ────────── ────────
  总计                 2.00       0.95       2.95        2
════════════════════════════════════════════════════════════
  💾 Token 用量报告已保存: test_report.json


'test_report.json'

In [40]:
def decorator(func):
    def inner():
        print("Before function call")
        result=func()
        print("After function call")
        return result
    return inner

@decorator
def say_hello():
    print("Hello, World!")
    return 10
b=say_hello()
print(b)

Before function call
Hello, World!
After function call
10


In [44]:
import sys
print(sys.path)
target='../scripts'
sys.path.insert(0, target)

['/data/hyj/miniconda/envs/biojson/lib/python311.zip', '/data/hyj/miniconda/envs/biojson/lib/python3.11', '/data/hyj/miniconda/envs/biojson/lib/python3.11/lib-dynload', '', '/data/hyj/miniconda/envs/biojson/lib/python3.11/site-packages']


In [53]:
import json
import os
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv
from token_tracker import TokenTracker
from text_utils import strip_references
load_dotenv()

# 1. 初始化客户端
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")  # 确保加上了 /v1
)

# Token 用量追踪
tracker = TokenTracker(model=os.getenv("MODEL", "Vendor2/Claude-4.6-opus"))

# 2. 路径配置 (从环境变量读取，run.sh 统一管理)
BASE_DIR = os.getenv("BASE_DIR", "/data/haotianwu/biojson")
INPUT_DIR = os.getenv("MD_DIR", os.path.join(BASE_DIR, "md"))
OUTPUT_DIR = os.getenv("RAW_EXTRACTIONS_DIR", os.path.join(BASE_DIR, "reports/raw_extractions"))
PROMPT_PATH = os.getenv("PROMPT_PATH", os.path.join(BASE_DIR, "configs/nutri_plant.txt"))
SCHEMA_PATH = os.getenv("SCHEMA_PATH", os.path.join(BASE_DIR, "configs/nutri_plant.json"))
EXTRACT_TOKENS_DIR = os.getenv("EXTRACT_TOKENS_DIR", os.path.join(BASE_DIR, "reports/extract_tokens"))

# 确保输出目录存在
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRACT_TOKENS_DIR, exist_ok=True)

# 3. 读取 Prompt 和 Schema
with open(PROMPT_PATH, 'r', encoding='utf-8') as f:
    base_prompt = f.read()

with open(SCHEMA_PATH, 'r', encoding='utf-8') as f:
    template = json.load(f)
    paradigm = template.get("CropNutrientMetabolismGeneExtraction", template)

# 4. 构建 function_calling 的 tool 定义
# 将基因字段的 schema 转为 function parameters 格式
GENE_PROPERTIES = {}
gene_def = paradigm.get("$defs", {}).get("NutrientMetabolismGene", {})
for field_name, field_schema in gene_def.get("properties", {}).items():
    # 简化 anyOf 为 string 类型（function_calling 不支持 anyOf）
    desc = field_schema.get("description", "")
    print(desc)
    if field_name == "Key_Intermediate_Metabolites_Affected" or field_name == "Interacting_Proteins":
        GENE_PROPERTIES[field_name] = {
            "type": "array",
            "items": {"type": "string"},
            "description": desc
        }
    else:
        GENE_PROPERTIES[field_name] = {
            "type": "string",
            "description": desc
        }


Gene symbol/name used in the paper (e.g., OsNAS2, ZmPSY1, SlGGP1, AtVTE2).
Accession/ID (NCBI/Ensembl/Phytozome/Gramene/LOC_*/AT*/locus tag) as reported.
Crop species common name (e.g., Rice, Maize, Wheat, Tomato).
Latin name (e.g., Oryza sativa, Zea mays, Triticum aestivum).
Cultivar/line used (e.g., Nipponbare, B73).
Reference genome assembly/version used for coordinates if reported.
Gene genomic interval or coordinate (e.g., Chr1:12345–13579). Exclude variant positions.
Controlled vocabulary recommended: Enzyme; Transporter; Storage/Binding; Regulatory_TF; Regulatory_RNA; Signaling; Other.
Protein family/domain (e.g., bHLH TF, SDR, cytochrome P450, MATE transporter).
If enzyme gene: enzyme name or biochemical function (e.g., phytoene synthase; GDP-L-galactose phosphorylase). If regulator: brief function (e.g., activates carotenoid pathway genes).
EC number if explicitly reported (e.g., 2.5.1.32).
The final nutritional product whose presence/yield/content is the terminal trait (e.g.,

In [2]:
from openai import OpenAI
import json

# 初始化客户端
# 注意：OpenAI 兼容接口通常需要在 base_url 后面加上 /v1
client = OpenAI(
    api_key="u0rgjezq53e5tv01000dg95v0yqn2eecv02b7z3r", 
    base_url="https://api.gpugeek.com/v1" 
)

try:
    # 发起简单的聊天请求
    response = client.chat.completions.create(
        model="Vendor2/Claude-4.5-Sonnet",
        messages=[{"role": "user", "content": "1+1=?"}]
    )
    
    # 成功时：提取回答内容
    # 注意：这里的 response 实际上可以像字典一样访问或转化
    print("✅ 请求成功！")
    print(f"回答内容: {response}")

except Exception as e:
    print("❌ 请求失败！")
    
    # --- 核心：提取报错原因的字典逻辑 ---
    # 大多数兼容 OpenAI 的 SDK 都会把详细报错存在 e.body 字典里
    if hasattr(e, 'body'):
        error_dict = e.body  # 这就是你想要的那个“报错字典”
        
        # 提取具体的 message
        # OpenAI 格式通常是：{"error": {"message": "...", "type": "..."}}
        error_msg = error_dict.get('error', {}).get('message', '未知错误信息')
        error_type = error_dict.get('error', {}).get('type', 'UnknownError')
        
        print(f"错误类型: {error_type}")
        print(f"报错原因: {error_msg}")
        
        # 如果你想看完整的字典长什么样，可以取消下面这行的注释
        # print(f"完整报错字典: {json.dumps(error_dict, indent=2, ensure_ascii=False)}")
    else:
        print(f"非 API 类型的错误: {e}")


✅ 请求成功！
回答内容: ChatCompletion(id='Vendor2-msg_bdrk_01V4kjVhc2T97S9sSWj9H5so', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1+1=2', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774002450, model='Vendor2/Claude-4.5-Sonnet', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=9, prompt_tokens=13, total_tokens=22, completion_tokens_details=None, prompt_tokens_details=None), metrics={'input_token_count': 13, 'output_token_count': 9, 'predict_time': 3.490240029})
